# Medición pendiente de BT en Google Colab

Este cuaderno compila el mismo proyecto C++ para Linux y ejecuta **un caso por vez**. Los tiempos de Colab son de otra máquina y deben identificarse como tales en la tabla. Ejecuta las celdas en orden y conserva los archivos de resultados.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()  # Selecciona testores_bt_colab.zip
assert 'testores_bt_colab.zip' in uploaded, 'Sube testores_bt_colab.zip'
with zipfile.ZipFile('testores_bt_colab.zip') as z:
    z.extractall('/content')
ROOT = Path('/content/testores_bt_colab')
assert (ROOT / 'CMakeLists.txt').is_file()
print('Proyecto extraído en:', ROOT)


In [ ]:
import subprocess
subprocess.run(['cmake', '-S', str(ROOT), '-B', str(ROOT / 'build-colab'), '-DCMAKE_BUILD_TYPE=Release', '-DTESTORES_BUILD_TESTS=OFF'], check=True)
subprocess.run(['cmake', '--build', str(ROOT / 'build-colab'), '--parallel', '2'], check=True)
EXE = ROOT / 'build-colab/testores'
assert EXE.is_file(), 'No se creó el ejecutable de Linux'
print('Ejecutable:', EXE)


## Ejecutar una medición

Cambia `CASO` a `phi4`, `phi5`, `gamma3` o `gamma4`, y `ORDEN` a `original` o `ascendente`. Son ocho combinaciones. No ejecutes dos mediciones a la vez. La celda imprime un aviso cada 30 segundos. No hay límite de tiempo dentro del cuaderno. Un corte de sesión de Colab deja el caso pendiente.


In [ ]:
import re, time, subprocess
from google.colab import files
CASO = 'phi4'          # phi4, phi5, gamma3 o gamma4
ORDEN = 'original'    # original o ascendente
assert CASO in ('phi4', 'phi5', 'gamma3', 'gamma4')
assert ORDEN in ('original', 'ascendente')
name = f'{CASO}_theta_AB'
matrix = ROOT / 'data' / (f'{name}.txt' if ORDEN == 'original' else f'ascendente/{name}_ascendente.txt')
assert matrix.is_file(), matrix
results = ROOT / 'resultados'
results.mkdir(exist_ok=True)
log = results / f'bt_{CASO}_{ORDEN}.txt'
cmd = [str(EXE), 'bt', '--entrada', str(matrix)]
print('Ejecutando:', ' '.join(cmd), flush=True)
print('Log:', log, flush=True)
started = time.monotonic()
with log.open('w', encoding='utf-8') as output:
    process = subprocess.Popen(cmd, cwd=ROOT, stdout=output, stderr=subprocess.STDOUT)
    try:
        while process.poll() is None:
            elapsed = time.monotonic() - started
            print(f'Sigue ejecutándose: {elapsed / 60:.1f} min', flush=True)
            time.sleep(30)
    except KeyboardInterrupt:
        process.kill()
        process.wait()
        print('Interrumpido: sin medición final.', flush=True)
text = log.read_text(encoding='utf-8', errors='replace')
match = re.search(r'Tiempo total de ejecuci[^\n]*?:\s*([0-9.]+)\s*(µs|ms|s)', text)
if process.returncode == 0 and match:
    value, unit = float(match.group(1)), match.group(2)
    ms = value / 1000 if unit == 'µs' else value * 1000 if unit == 's' else value
    print(f'BT {CASO} {ORDEN}: {value:.3f} {unit} = {ms:.3f} ms')
else:
    print(f'Sin medición final (código {process.returncode}). Revisa el log.')
files.download(str(log))


## Ejecutar las cuatro matrices automáticamente

La siguiente celda ejecuta las **ocho combinaciones** (cuatro matrices, dos órdenes) de forma consecutiva. Usa esta opción **en vez de** repetir la celda individual. No hay límite de tiempo dentro del cuaderno. Al terminar descarga un ZIP con el resumen CSV y los registros. Si Colab corta la sesión antes, solo se conservan los resultados que hayas descargado previamente.


In [ ]:
import csv, re, subprocess, time, zipfile
from google.colab import files
results = ROOT / 'resultados'
results.mkdir(exist_ok=True)
rows = []
pairs = [(case, order) for case in ('phi4', 'phi5', 'gamma3', 'gamma4') for order in ('original', 'ascendente')]
for case, order in pairs:
    name = f'{case}_theta_AB'
    matrix = ROOT / 'data' / (f'{name}.txt' if order == 'original' else f'ascendente/{name}_ascendente.txt')
    assert matrix.is_file(), matrix
    log = results / f'bt_{case}_{order}.txt'
    cmd = [str(EXE), 'bt', '--entrada', str(matrix)]
    print(f'Iniciando {case} / {order}', flush=True)
    started = time.monotonic()
    with log.open('w', encoding='utf-8') as output:
        process = subprocess.Popen(cmd, cwd=ROOT, stdout=output, stderr=subprocess.STDOUT)
        try:
            while process.poll() is None:
                elapsed = time.monotonic() - started
                print(f'  {case} / {order}: {elapsed / 60:.1f} min', flush=True)
                time.sleep(30)
        except KeyboardInterrupt:
            process.kill()
            process.wait()
            raise
    output_text = log.read_text(encoding='utf-8', errors='replace')
    match = re.search(r'Tiempo total de ejecuci[^\n]*?:\s*([0-9.]+)\s*(µs|ms|s)', output_text)
    if process.returncode == 0 and match:
        value, unit = float(match.group(1)), match.group(2)
        ms = value / 1000 if unit == 'µs' else value * 1000 if unit == 's' else value
        status = 'completo'
        print(f'  Terminado: {value:.3f} {unit} = {ms:.3f} ms', flush=True)
    else:
        ms = ''
        status = f'error_{process.returncode}'
        print(f'  Sin medición final: {status}', flush=True)
    rows.append((case, order, status, ms, log.name))
    with (results / 'resumen_bt_colab.csv').open('w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(('matriz', 'orden', 'estado', 'tiempo_ms', 'registro'))
        writer.writerows(rows)
archive = ROOT / 'resultados_bt_colab.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in sorted(results.glob('*')):
        z.write(path, path.name)
print('Resumen:', results / 'resumen_bt_colab.csv')
files.download(str(archive))
